In [ ]:
from openff.toolkit import Molecule, Topology, ForceField
from openff.interchange import Interchange
from openff.units import unit, Quantity
from openff.toolkit.utils.toolkits import NAGLToolkitWrapper
import numpy as np
import time
import mdtraj as md
import openmm
from openff.toolkit.typing.engines.smirnoff import ForceField
from openff.toolkit.utils import get_data_file_path
from pandas import read_csv
from openff.units.openmm import to_openmm
from openff.interchange.components._packmol import pack_box

In [ ]:
# input smiles for desired organic solvent
# if using more than one organic solvent, copy these lines and fill out same information specific to the new organic solvent
monomer =Molecule.from_smiles("")
monomer.generate_conformers(n_conformers=1)

In [ ]:
# input smiles for water, use if calculating mixture of organic solvent and water
water = Molecule.from_smiles("O")
water.generate_conformers(n_conformers=1)

In [ ]:
# creates a box with coordinates: [value, 0.00, 0.00][0.00, value, 0.00][0.00, 0.00, value] angstroms
cubic_box =unit.Quantity (value * np.eye(3), unit.angstrom)

# enter number of molecules you want to simulate
# must have the same number of total molecules for each simulation if you are calculating enthalpy of mixing
# keep mole fraction in mind when deciding on the number of molecules to use
n_monomer = 
n_water = 

# packs the box with the amount of desired molecules
# add other molecules if using more than one organic solvent and water
from openff.interchange.components._packmol import pack_box
packed_topology = pack_box(
    molecules=[monomer, water],
    number_of_copies=[n_monomer, n_water],
    solute=None,
    tolerance=2.0*unit.angstrom,
    box_vectors=cubic_box,
)

In [ ]:
from openmm import MonteCarloBarostat
from openmm.unit import atmosphere, kelvin

packed_interchange = Interchange.from_smirnoff(
            force_field=ForceField("openff-2.2.0.offxml", "tip3p.offxml"),
            topology=packed_topology, box=cubic_box)            
system = packed_interchange.to_openmm()

# input desired pressure (atm), temperature (K), and friction factor
# to change units, import the new unit above
barostat = MonteCarloBarostat(
    1.0 * atmosphere,
    300.0 * kelvin,
    25,
)

# minimizes file and saves topology to a new pdb file
system.addForce(barostat)
packed_interchange.minimize()
packed_interchange.to_pdb("filename.pdb")


/projects/ladu6977/polyzymd/.pixi/envs/cuda-12-4/lib/python3.11/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):


In [20]:
# Length of the simulation.
num_steps = 5000000  # number of integration steps to run

# Logging options.
trj_freq = 16667  # number of steps per written trajectory frame
data_freq = 5000  # number of steps per written simulation statistics

# Integration options
time_step = 2 * openmm.unit.femtoseconds  # simulation timestep
temperature = 300 * openmm.unit.kelvin  # simulation temperature
friction = 1 / openmm.unit.picosecond  # friction constant

integrator = openmm.LangevinMiddleIntegrator(temperature, friction, time_step)

In [ ]:

# Create Simulation
simulation = openmm.app.Simulation(
    packed_interchange.to_openmm_topology(),
    system,
    integrator,
)

# Set positions from the Interchange object
simulation.context.setPositions(
    to_openmm(packed_interchange.positions)
)
simulation.context.setVelocitiesToTemperature(temperature)

# creates a dcd file for the trajectory and a csv file for the simulation data
# *does not output energy per molecule*, must divide energy by the number of total molecules for true kJ/mole value if using in a calculation
dcd_reporter =openmm.app.DCDReporter("filename_trajectory.dcd", trj_freq)
state_data_reporter = openmm.app.StateDataReporter(
    "filename_data.csv",
    data_freq,
    step=True,
    potentialEnergy=True,
    kineticEnergy=True,
    totalEnergy=True,
    temperature=True,
    volume=True,
    density=True,
    speed=True,
    time=True,
    elapsedTime=True,
    
)
simulation.reporters.append(dcd_reporter)
simulation.reporters.append(state_data_reporter)

In [22]:
print("Starting simulation")
start = time.process_time()

# Run the simulation
simulation.step(num_steps)

end = time.process_time()
print(f"Elapsed time {end - start} seconds")
print("Done!")

Starting simulation
Elapsed time 1374.46599803 seconds
Done!
